# Video Clone — Colab remote worker

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daokimluc/video-clone-colab/blob/main/video_clone_colab_backend.ipynb)

Repo: https://github.com/daokimluc/video-clone-colab

## Cách dùng (desktop Video Clone)
1. **Runtime → Run all**
2. Cell **Tunnel** in `REMOTE_URL` (`https://….trycloudflare.com`) + `SHARED_SECRET`
3. App → **Cấu hình → Remote**: dán cả URL và secret
4. Giữ runtime Colab đang chạy

Nếu gặp `address already in use` trên port 8765: notebook sẽ **tự kill port** rồi start lại. Hoặc **Runtime → Restart session** rồi Run all.

## 1) Cài dependency + cloudflared

In [ ]:
!pip -q install fastapi uvicorn httpx

import os, shutil

def ensure_cloudflared():
    if shutil.which('cloudflared'):
        print('cloudflared already on PATH:', shutil.which('cloudflared'))
        return
    print('Installing cloudflared…')
    url = 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    dest = '/usr/local/bin/cloudflared'
    !wget -q -O /tmp/cloudflared {url}
    !chmod +x /tmp/cloudflared
    !cp /tmp/cloudflared {dest}
    print('cloudflared installed:', shutil.which('cloudflared') or dest)

ensure_cloudflared()
!cloudflared --version

## 2) Shared secret (dán vào app)

In [ ]:
import secrets

SHARED_SECRET = secrets.token_urlsafe(16)
print('=' * 60)
print('SHARED_SECRET (copy vào app → Shared secret):')
print(SHARED_SECRET)
print('=' * 60)

## 3) Start FastAPI worker

Tự giải phóng port nếu còn process cũ (`Errno 98 address already in use`).

In [ ]:
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
import uvicorn
import threading
import time
import socket
import subprocess
import os
import httpx

PREFERRED_PORT = 8765

def port_in_use(port: int, host: str = '127.0.0.1') -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0

def free_port(port: int) -> None:
    """Kill listeners on port (Linux Colab). Safe no-op if nothing bound."""
    if not port_in_use(port):
        print(f'Port {port} is free')
        return
    print(f'Port {port} busy — freeing…')
    # fuser is available on most Colab images
    subprocess.run(['bash', '-lc', f'fuser -k {port}/tcp || true'], check=False)
    time.sleep(0.8)
    if port_in_use(port):
        # fallback: lsof
        subprocess.run(
            ['bash', '-lc', f"pids=$(lsof -t -i:{port} 2>/dev/null); [ -n \"$pids\" ] && kill -9 $pids || true"],
            check=False,
        )
        time.sleep(0.8)
    if port_in_use(port):
        print(f'WARNING: port {port} still busy after kill')
    else:
        print(f'Port {port} freed')

def pick_port(start: int = PREFERRED_PORT, tries: int = 20) -> int:
    free_port(start)
    if not port_in_use(start):
        return start
    for p in range(start + 1, start + tries):
        if not port_in_use(p):
            print(f'Using alternate port {p}')
            return p
    raise RuntimeError(f'No free port near {start}')

WORKER_PORT = pick_port(PREFERRED_PORT)

app = FastAPI(title='VideoClone Colab Worker')

def check(secret: str | None):
    if secret != SHARED_SECRET:
        raise HTTPException(401, 'bad secret')

@app.get('/health')
def health(x_vc_secret: str | None = Header(default=None)):
    check(x_vc_secret)
    return {'ok': True, 'worker': 'colab', 'port': WORKER_PORT}

@app.get('/capabilities')
def caps(x_vc_secret: str | None = Header(default=None)):
    check(x_vc_secret)
    return {'asr': True, 'tts': True, 'worker': 'colab'}

class AsrReq(BaseModel):
    note: str = 'stub — wire faster-whisper here'

@app.post('/infer/asr')
def infer_asr(body: AsrReq, x_vc_secret: str | None = Header(default=None)):
    check(x_vc_secret)
    return {
        'status': 'not_implemented',
        'hint': 'Desktop still runs ASR locally until infer is wired',
        'note': body.note,
    }

def _run_uvicorn():
    uvicorn.run(app, host='127.0.0.1', port=WORKER_PORT, log_level='warning')

# Always (re)bind after free_port — do not skip when re-running cell
if globals().get('_VC_WORKER_THREAD') and globals().get('_VC_WORKER_THREAD').is_alive():
    print('Previous worker thread still alive; port was freed so starting a new bind on', WORKER_PORT)

_VC_WORKER_THREAD = threading.Thread(target=_run_uvicorn, daemon=True)
_VC_WORKER_THREAD.start()
time.sleep(1.5)

# local self-check (retry a few times)
ok = False
for i in range(10):
    try:
        r = httpx.get(
            f'http://127.0.0.1:{WORKER_PORT}/health',
            headers={'X-VC-Secret': SHARED_SECRET},
            timeout=3.0,
        )
        print('Local worker health:', r.status_code, r.json())
        ok = r.status_code == 200
        break
    except Exception as e:
        time.sleep(0.4)
        last_err = e
if not ok:
    print('Local worker not ready:', last_err if 'last_err' in dir() else 'unknown')
    print('Try: Runtime → Restart session, then Run all')
else:
    print(f'Worker OK → http://127.0.0.1:{WORKER_PORT}')
    print('Next: run Tunnel cell (uses WORKER_PORT)')

## 4) Cloudflare quick tunnel → **REMOTE_URL**

Dùng đúng `WORKER_PORT` từ cell worker (mặc định 8765).

In [ ]:
import re
import subprocess
import time
import shutil
import httpx

assert shutil.which('cloudflared'), 'cloudflared missing — re-run install cell'
assert 'SHARED_SECRET' in globals(), 'Run SHARED_SECRET cell first'
assert 'WORKER_PORT' in globals(), 'Run worker cell first'

# stop previous tunnel if re-run
if globals().get('_VC_TUNNEL_PROC') is not None:
    try:
        _VC_TUNNEL_PROC.terminate()
        time.sleep(0.5)
    except Exception:
        pass

cmd = ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{WORKER_PORT}']
print('Starting:', ' '.join(cmd))
_VC_TUNNEL_PROC = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

REMOTE_URL = None
pattern = re.compile(r'https://[a-zA-Z0-9.-]+\.trycloudflare\.com')
deadline = time.time() + 90
lines = []

while time.time() < deadline and REMOTE_URL is None:
    line = _VC_TUNNEL_PROC.stdout.readline()
    if not line:
        if _VC_TUNNEL_PROC.poll() is not None:
            break
        time.sleep(0.1)
        continue
    line = line.rstrip()
    lines.append(line)
    m = pattern.search(line)
    if m:
        REMOTE_URL = m.group(0).rstrip('/')
        break

if not REMOTE_URL:
    print('--- last cloudflared lines ---')
    print('\n'.join(lines[-40:]))
    raise RuntimeError('Không bắt được URL trycloudflare.com — chạy lại cell tunnel')

print('=' * 60)
print('REMOTE_URL (copy vào app → Remote URL):')
print(REMOTE_URL)
print('SHARED_SECRET (copy vào app → Shared secret):')
print(SHARED_SECRET)
print('WORKER_PORT:', WORKER_PORT)
print('=' * 60)
print('App: Backend mode = Remote')

try:
    time.sleep(1.5)
    hr = httpx.get(
        REMOTE_URL + '/health',
        headers={'X-VC-Secret': SHARED_SECRET},
        timeout=20.0,
        follow_redirects=True,
    )
    print('Public /health via tunnel:', hr.status_code, hr.text[:300])
except Exception as e:
    print('Public health probe failed (tunnel warming?):', e)

print('Giữ runtime Colab + cell này. Disconnect = tunnel chết.')

## Troubleshooting

| Lỗi | Cách xử lý |
|-----|------------|
| `address already in use` / Errno 98 | Cell worker đã tự `fuser -k 8765/tcp`. Nếu vẫn lỗi: **Runtime → Restart session** → Run all |
| Không thấy REMOTE_URL | Chạy lại cell Tunnel; kiểm tra cloudflared install |
| App readiness fail | Dán **cả** REMOTE_URL + SHARED_SECRET; mode = Remote |
| Tunnel chết sau vài phút | Colab idle disconnect — re-run Tunnel cell, URL **đổi** |